In [1]:
# Install the official Kaggle API client
!pip install kaggle

In [2]:
from google.colab import files

print("Please select your 'kaggle.json' file to upload:")
files.upload()
# A file selector box will appear. Click 'Choose Files' and select the kaggle.json file.

Please select your 'kaggle.json' file to upload:


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gangulasomashekar","key":"324fab117239ff9bc1b5060e1a2215c8"}'}

In [3]:
import os

# 1. Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# 2. Move the uploaded kaggle.json file into the .kaggle directory
!mv kaggle.json ~/.kaggle/

# 3. Set permissions: The file must be read-only for security (owner only)
!chmod 600 ~/.kaggle/kaggle.json

print("\nKaggle API Key setup complete! You are now authenticated.")

# Verify the file is in place and permissions are correct (optional)
!ls -l ~/.kaggle/


Kaggle API Key setup complete! You are now authenticated.
total 4
-rw------- 1 root root 73 Nov 11 17:55 kaggle.json


In [4]:
import os

# 1. Define the desired folder name
DOWNLOAD_PATH = './clinical_data'

# 2. Create the folder if it doesn't exist
# The -p flag in !mkdir ensures no error is thrown if the directory already exists
!mkdir -p {DOWNLOAD_PATH}
print(f"Directory '{DOWNLOAD_PATH}' created.")

# 3. Download the dataset into the specified folder using the -p argument
# The -d flag specifies the dataset, and the -p flag specifies the path.
!kaggle datasets download -d azmayensabil/doctor-patient-conversation-large -p {DOWNLOAD_PATH}
print("Download complete.")

# 4. Unzip the downloaded file inside the target folder
# The zip file will be located at: ./clinical_data/doctor-patient-conversation-large.zip
ZIP_FILE_PATH = os.path.join(DOWNLOAD_PATH, 'doctor-patient-conversation-large.zip')

# -q for quiet (optional), -d for destination directory
!unzip -q {ZIP_FILE_PATH} -d {DOWNLOAD_PATH}
print(f"Unzip complete. Files are extracted to: {DOWNLOAD_PATH}")

# 5. (Optional) List the contents of the folder to confirm the download and unzip
print("\n--- Folder Contents ---")
!ls {DOWNLOAD_PATH}

Directory './clinical_data' created.
Dataset URL: https://www.kaggle.com/datasets/azmayensabil/doctor-patient-conversation-large
License(s): unknown
  0% 0.00/786k [00:00<?, ?B/s]
100% 786k/786k [00:00<00:00, 1.04GB/s]
Download complete.
Unzip complete. Files are extracted to: ./clinical_data

--- Folder Contents ---
CAR0001.txt			       RES0010.txt  RES0081.txt  RES0151.txt
CAR0002.txt			       RES0011.txt  RES0082.txt  RES0152.txt
CAR0003.txt			       RES0012.txt  RES0083.txt  RES0153.txt
CAR0004.txt			       RES0013.txt  RES0084.txt  RES0154.txt
CAR0005.txt			       RES0014.txt  RES0085.txt  RES0155.txt
DER0001.txt			       RES0015.txt  RES0086.txt  RES0156.txt
doctor-patient-conversation-large.zip  RES0016.txt  RES0087.txt  RES0158.txt
GAS0001.txt			       RES0017.txt  RES0088.txt  RES0159.txt
GAS0002.txt			       RES0018.txt  RES0089.txt  RES0160.txt
GAS0003.txt			       RES0019.txt  RES0090.txt  RES0161.txt
GAS0004.txt			       RES0020.txt  RES0091.txt  RES0162.txt
GAS0005.txt			

In [5]:
pip install transformers torch datasets seqeval scikit-learn pandas tqdm evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=633def35ba4f666d55cda1c2256bd15f7d8070fbf7d2e5c52e0a08bfb4559c54
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [6]:
import os
import sys
import json
import re
import argparse
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, Counter
from datetime import datetime
from dataclasses import dataclass
from tqdm import tqdm
import pandas as pd

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Check imports
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForTokenClassification,
        pipeline
    )
except ImportError as e:
    logger.error(f"Missing dependencies: {e}")
    print("❌ Missing dependencies. Please install:")
    print("\npip install transformers torch pandas tqdm\n")
    sys.exit(1)

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class ModelConfig:
    """Model configuration parameters"""
    # Using a pre-trained medical NER model that works well out-of-the-box
    model_name: str = "samrawal/bert-base-uncased_clinical-ner"
    # Alternative models you can try:
    # - "d4data/biomedical-ner-all"
    # - "bvanaken/clinical-ner-biobert"
    # - "emilyalsentzer/Bio_ClinicalBERT"
    max_length: int = 512
    batch_size: int = 16

class Config:
    def __init__(self, data_dir: str = "/content/clinical_data"):
        self.project_root = Path.cwd()
        self.raw_data_dir = Path(data_dir)
        self.output_dir = self.project_root / "outputs"

        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        (self.output_dir / "summaries").mkdir(exist_ok=True)
        (self.output_dir / "entities").mkdir(exist_ok=True)
        (self.output_dir / "statistics").mkdir(exist_ok=True)

        self.model = ModelConfig()

# ============================================================================
# CONVERSATION PARSER
# ============================================================================

class ConversationParser:
    def __init__(self, config: Config):
        self.config = config

    def parse_text(self, text: str, filename: str = "conversation.txt") -> Dict:
        """Parse conversation text"""
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        turns = []

        current_speaker = None
        current_text = []

        for line in lines:
            line_clean = line.strip()

            if line_clean.startswith('D:'):
                if current_speaker and current_text:
                    turns.append({
                        'speaker': current_speaker,
                        'text': ' '.join(current_text)
                    })
                current_speaker = 'doctor'
                current_text = [line_clean[2:].strip()]
            elif line_clean.startswith('P:'):
                if current_speaker and current_text:
                    turns.append({
                        'speaker': current_speaker,
                        'text': ' '.join(current_text)
                    })
                current_speaker = 'patient'
                current_text = [line_clean[2:].strip()]
            elif current_speaker and line_clean:
                current_text.append(line_clean)

        if current_speaker and current_text:
            turns.append({
                'speaker': current_speaker,
                'text': ' '.join(current_text)
            })

        return {
            'filename': filename,
            'turns': turns,
            'num_turns': len(turns)
        }

# ============================================================================
# ENHANCED ENTITY POST-PROCESSOR
# ============================================================================

class ClinicalEntityPostProcessor:
    """Advanced entity cleaning and categorization for clinical text"""

    def __init__(self):
        # Map model entities to our categories
        self.entity_mapping = {
            # Symptoms
            'problem': 'SYMPTOM',
            'symptom': 'SYMPTOM',
            'sign': 'SYMPTOM',
            'disease': 'SYMPTOM',
            'disorder': 'SYMPTOM',

            # Medications
            'drug': 'MEDICATION',
            'medication': 'MEDICATION',
            'treatment': 'MEDICATION',

            # Diagnoses
            'diagnosis': 'DIAGNOSIS',
            'condition': 'DIAGNOSIS',

            # Tests
            'test': 'TEST',
            'procedure': 'TEST',

            # Body parts
            'anatomy': 'BODY_PART',
            'body_part': 'BODY_PART'
        }

        # Common clinical stop words
        self.stop_words = {
            'patient', 'doctor', 'history', 'normal', 'negative', 'positive',
            'like', 'just', 'get', 'got', 'going', 'know', 'think', 'maybe',
            'probably', 'possibly', 'seems', 'appears', 'looks', 'feels'
        }

        # Medical term variations and corrections
        self.medical_corrections = {
            'c/o': 'complains of',
            's/o': 'suggestive of',
            'r/o': 'rule out',
            'h/o': 'history of',
            'coughing': 'cough',
            'vomiting': 'vomit',
            'nauseous': 'nausea',
            'aching': 'ache',
            'hurts': 'pain',
            'hurting': 'pain'
        }

    def clean_and_categorize_entities(self, raw_entities: List[Dict]) -> List[Dict]:
        """Clean and categorize entities from the NER model"""
        cleaned_entities = []

        for entity in raw_entities:
            # Clean the entity text
            entity_text = self._clean_entity_text(entity['word'])

            if not entity_text or len(entity_text) < 2:
                continue

            # Skip stop words
            if entity_text.lower() in self.stop_words:
                continue

            # Map to our entity categories
            entity_type = self._map_entity_type(entity['entity_group'])

            if entity_type:
                cleaned_entity = {
                    'word': entity_text,
                    'entity_group': entity_type,
                    'score': float(entity['score']),
                    'original_entity': entity['entity_group']
                }
                cleaned_entities.append(cleaned_entity)

        # Remove duplicates and merge similar entities
        final_entities = self._deduplicate_entities(cleaned_entities)

        return final_entities

    def _clean_entity_text(self, text: str) -> str:
        """Clean entity text"""
        # Remove subword markers and extra spaces
        cleaned = text.replace('##', '').strip()

        # Apply medical corrections
        cleaned_lower = cleaned.lower()
        if cleaned_lower in self.medical_corrections:
            cleaned = self.medical_corrections[cleaned_lower]

        # Remove common prefixes/suffixes
        cleaned = re.sub(r'^(mild|moderate|severe|acute|chronic)\s+', '', cleaned)
        cleaned = re.sub(r'\s+(disease|disorder|syndrome|pain|ache)$', '', cleaned)

        # Capitalize properly
        if len(cleaned) > 1:
            cleaned = cleaned[0].upper() + cleaned[1:]

        return cleaned.strip()

    def _map_entity_type(self, original_type: str) -> str:
        """Map model entity types to our categories"""
        original_lower = original_type.lower()

        for key, value in self.entity_mapping.items():
            if key in original_lower:
                return value

        # Default mapping based on common patterns
        if any(word in original_lower for word in ['drug', 'med', 'pill']):
            return 'MEDICATION'
        elif any(word in original_lower for word in ['symptom', 'pain', 'fever']):
            return 'SYMPTOM'
        elif any(word in original_lower for word in ['diagnosis', 'disease']):
            return 'DIAGNOSIS'
        elif any(word in original_lower for word in ['test', 'exam']):
            return 'TEST'
        elif any(word in original_lower for word in ['body', 'anatomy']):
            return 'BODY_PART'

        return 'SYMPTOM'  # Default fallback

    def _deduplicate_entities(self, entities: List[Dict]) -> List[Dict]:
        """Remove duplicate and very similar entities"""
        seen = set()
        unique_entities = []

        for entity in entities:
            # Create a key based on cleaned text and type
            key = (entity['word'].lower(), entity['entity_group'])

            if key not in seen:
                seen.add(key)
                unique_entities.append(entity)

        return unique_entities

# ============================================================================
# ADVANCED INFERENCE ENGINE
# ============================================================================

class ClinicalNERPipeline:
    """Advanced clinical NER pipeline using pre-trained models"""

    def __init__(self, config: Config):
        self.config = config
        self.postprocessor = ClinicalEntityPostProcessor()
        self.ner_pipeline = None
        self._load_model()

    def _load_model(self):
        """Load the pre-trained clinical NER model"""
        try:
            logger.info(f"🚀 Loading clinical NER model: {self.config.model.model_name}")

            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1,
                batch_size=self.config.model.batch_size
            )

            logger.info("✅ Clinical NER model loaded successfully!")

        except Exception as e:
            logger.error(f"❌ Error loading model: {e}")
            logger.info("🔄 Trying fallback model...")

            # Fallback to a more reliable model
            self.config.model.model_name = "emilyalsentzer/Bio_ClinicalBERT"
            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1
            )

    def extract_clinical_entities(self, conversation: Dict) -> Dict:
        """Extract clinical entities from conversation"""
        all_entities = []

        for turn in conversation['turns']:
            if not turn['text'].strip():
                continue

            try:
                # Extract entities from this turn
                raw_entities = self.ner_pipeline(turn['text'])

                # Clean and categorize entities
                cleaned_entities = self.postprocessor.clean_and_categorize_entities(raw_entities)

                # Add conversation context
                for entity in cleaned_entities:
                    entity['speaker'] = turn['speaker']
                    entity['turn_text'] = turn['text'][:100] + '...' if len(turn['text']) > 100 else turn['text']
                    entity['confidence'] = entity['score']

                all_entities.extend(cleaned_entities)

            except Exception as e:
                logger.warning(f"⚠️ Error processing turn: {str(e)[:100]}...")
                continue

        return {
            'filename': conversation['filename'],
            'entities': all_entities,
            'num_entities': len(all_entities),
            'speaker_breakdown': self._get_speaker_breakdown(all_entities)
        }

    def _get_speaker_breakdown(self, entities: List[Dict]) -> Dict[str, int]:
        """Get entity count by speaker"""
        breakdown = defaultdict(int)
        for entity in entities:
            breakdown[entity.get('speaker', 'unknown')] += 1
        return dict(breakdown)

# ============================================================================
# INTELLIGENT RESULTS GENERATOR
# ============================================================================

class ClinicalResultsGenerator:
    """Generate clean, meaningful clinical summaries"""

    @staticmethod
    def generate_detailed_results(extraction_result: Dict) -> Dict:
        """Generate detailed, organized results"""
        entities = extraction_result['entities']

        # Group entities by type
        symptoms = []
        medications = []
        diagnoses = []
        tests = []
        body_parts = []

        for entity in entities:
            entity_type = entity['entity_group']
            entity_text = entity['word']

            # Skip low confidence entities
            if entity.get('confidence', 0) < 0.6:
                continue

            if entity_type == 'SYMPTOM':
                symptoms.append(entity_text)
            elif entity_type == 'MEDICATION':
                medications.append(entity_text)
            elif entity_type == 'DIAGNOSIS':
                diagnoses.append(entity_text)
            elif entity_type == 'TEST':
                tests.append(entity_text)
            elif entity_type == 'BODY_PART':
                body_parts.append(entity_text)

        # Remove duplicates and sort
        symptoms = sorted(list(set(symptoms)))
        medications = sorted(list(set(medications)))
        diagnoses = sorted(list(set(diagnoses)))
        tests = sorted(list(set(tests)))
        body_parts = sorted(list(set(body_parts)))

        return {
            'symptoms': symptoms,
            'medications': medications,
            'diagnoses': diagnoses,
            'tests': tests,
            'body_parts': body_parts,
            'summary': {
                'total_entities': len(symptoms) + len(medications) + len(diagnoses) + len(tests) + len(body_parts),
                'symptoms_count': len(symptoms),
                'medications_count': len(medications),
                'diagnoses_count': len(diagnoses),
                'tests_count': len(tests),
                'body_parts_count': len(body_parts)
            }
        }

    @staticmethod
    def generate_clinical_summary(detailed_results: Dict) -> str:
        """Generate a clean, professional clinical summary"""
        summary = []
        summary.append("=" * 70)
        summary.append("CLINICAL ENTITY EXTRACTION SUMMARY")
        summary.append("=" * 70)

        # Patient Symptoms
        summary.append("\n🧬 PATIENT SYMPTOMS:")
        summary.append("-" * 40)
        symptoms = detailed_results['symptoms']
        if symptoms:
            for symptom in symptoms:
                summary.append(f"  • {symptom}")
        else:
            summary.append("  No symptoms documented")

        # Medications
        summary.append("\n💊 MEDICATIONS & TREATMENTS:")
        summary.append("-" * 40)
        medications = detailed_results['medications']
        if medications:
            for med in medications:
                summary.append(f"  • {med}")
        else:
            summary.append("  No medications mentioned")

        # Clinical Diagnoses
        summary.append("\n🏥 CLINICAL IMPRESSIONS:")
        summary.append("-" * 40)
        diagnoses = detailed_results['diagnoses']
        if diagnoses:
            for diag in diagnoses:
                summary.append(f"  • {diag}")
        else:
            summary.append("  No diagnoses documented")

        # Diagnostic Tests
        summary.append("\n🔬 DIAGNOSTIC TESTS:")
        summary.append("-" * 40)
        tests = detailed_results['tests']
        if tests:
            for test in tests:
                summary.append(f"  • {test}")
        else:
            summary.append("  No tests ordered")

        # Body Parts
        summary.append("\n📍 BODY PARTS EXAMINED:")
        summary.append("-" * 40)
        body_parts = detailed_results['body_parts']
        if body_parts:
            for part in body_parts:
                summary.append(f"  • {part}")
        else:
            summary.append("  No specific body parts mentioned")

        # Statistics
        summary.append("\n📊 EXTRACTION STATISTICS:")
        summary.append("-" * 40)
        stats = detailed_results['summary']
        summary.append(f"  Total clinical entities: {stats['total_entities']}")
        summary.append(f"  Symptoms identified: {stats['symptoms_count']}")
        summary.append(f"  Medications mentioned: {stats['medications_count']}")
        summary.append(f"  Clinical impressions: {stats['diagnoses_count']}")
        summary.append(f"  Diagnostic tests: {stats['tests_count']}")

        summary.append("=" * 70)
        return "\n".join(summary)

# ============================================================================
# MAIN PROCESSING PIPELINE
# ============================================================================

def process_clinical_conversation(conversation_text: str, filename: str = "conversation.txt") -> Dict:
    """Main pipeline to process clinical conversations"""

    # Initialize configuration
    config = Config()

    # Step 1: Parse conversation
    logger.info("📝 Parsing clinical conversation...")
    parser = ConversationParser(config)
    conversation = parser.parse_text(conversation_text, filename)
    logger.info(f"   Parsed {conversation['num_turns']} conversation turns")

    # Step 2: Extract clinical entities using pre-trained model
    logger.info("🔍 Extracting clinical entities...")
    ner_pipeline = ClinicalNERPipeline(config)
    extraction_result = ner_pipeline.extract_clinical_entities(conversation)
    logger.info(f"   Found {extraction_result['num_entities']} clinical entities")

    # Step 3: Generate clean results
    logger.info("📊 Generating clinical summary...")
    detailed_results = ClinicalResultsGenerator.generate_detailed_results(extraction_result)
    clinical_summary = ClinicalResultsGenerator.generate_clinical_summary(detailed_results)

    # Save results
    output_file = config.output_dir / "summaries" / f"{Path(filename).stem}_clinical_summary.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(clinical_summary)

    # Save entities as JSON
    entities_file = config.output_dir / "entities" / f"{Path(filename).stem}_entities.json"
    with open(entities_file, 'w', encoding='utf-8') as f:
        json.dump(extraction_result, f, indent=2, ensure_ascii=False)

    logger.info(f"✅ Results saved to: {output_file}")

    return {
        'extraction_result': extraction_result,
        'detailed_results': detailed_results,
        'clinical_summary': clinical_summary
    }

# ============================================================================
# SAMPLE CONVERSATION & EXECUTION
# ============================================================================

# Your sample conversation
SAMPLE_CONVERSATION = """D: How may I help you?

P: Hi, um, I'm here because I, I had this cough for a couple days and now my voice still hasn't recovered. It sounds really hoarse and I can't speak as well, so I was just wondering if you can give me something for that.

D: Possibly we will see just gotta ask a few questions first, and we'll, we'll do a physical exam as well afterwards, but certainly, we'll look into what is causing your symptoms. So you mentioned a cough. When did that exactly start?

P: That started three days ago.

D: Okay, and have you had a cough before?

P: Um, like before the three days you mean?

D: Yeah.

P: Um, no, I was pretty healthy before then.

D: I see and uh, with the cough, are you producing any phlegm or sputum?

P: No, it's pretty dry.

D: Okay, have you noticed any blood?

P: No.

D: Alright, and have you had a wheeze?

P: No, uh, no, nothing like that.

D: Alright, and have you had any uh, chest pain with the cough?

P: No.

D: Alright uh, and you noticed that your voice is, voice changed, and that was about 3 days ago as well?

P: Yeah, that, so, well no, actually that started about yesterday. Yeah, yesterday in the afternoon.

D: Okay, and uh, any triggers for this? Like were you at any events that you had to be uh, kind of speaking loudly or talking a lot or anything where you're straining your voice?

P: Um no, no, I wasn't. I had school, I came home, I didn't do anything like that.

D: Okay. Yeah, so it sounds, yeah, just sort of came on on its own. Alright, and have you had any like, eye redness, or discharge, or a runny nose?

P: I have had uh, I've had a runny nose before, but no eye discharge.

D: Okay. With the runny nose, what, what, what could you describe the uh, mucus that was coming out?

P: Uh, it was, it was clear.

D: And has that gone away now and then when was the runny nose?

P: Yeah, that's gone, yeah that's gone. That went away but like, um, yeah, two days ago maybe.

D: I see, and have you had a sore throat or do you have a sore throat?

P: Um I have, so, it hurts because I'm coughing, but it doesn't seem like it's my actual throat.

D: Alright, is there any pain with swallowing food or liquids?

P: No.

D: Okay. Um, and have you had any fevers or chills?

P: No.

D: Or any, any night sweats?

P: No.

D: And any changes to your weight recently?

P: Um, I've had some weight gain over the past like six months.

D: Okay, well that's great. And about how much?

P: Uh just, maybe like 5, 10 pounds. 5 to 10 pounds, somewhere around there.

D: Okay, well that's good. And um, have you had any nausea or vomiting?

P: No.

D: How about any abdominal pain?

P: Uh no.

D: Any uh, diarrhea?

P: No.

D: Or any urinary problems?

P: No.

D: Have you had any muscle aches or, or joint pains?

P: Uh, no.

D: Okay, and have you had any loss of your taste or sense of smell?

P: No.

D: Okay, um, and any skin changes or rashes?

P: Nope, nothing like that.

D: Okay, uh, and um, so you've been experiencing, so you had a runny nose that's gone away um, and now you've been having this cough for the past, dry cough, past three days and lost your voice yesterday. Have you been experiencing any other symptoms?

P: Um no, no other symptoms.

D: Alright, in the past, have you been diagnosed with any medical conditions?

P: Uh no, I've been pretty healthy.

D: That's great, and u, have you been, um, have you had any allergies before?

P: No.

D: Okay, and uh, do you take any medications regularly?

P: Uh, I just take some multivitamins.

D: Okay, and uh are you aware if your immunizations are up to date?

P: Uh, so I have everything except for that HPV vaccine.

D: Okay, um, is it, do you have a plan to get that one or?

P: I think so. I'm just uh, waiting to follow up with my family doctor about that.

D: Okay, well that, that's a good idea for sure. So it's great that you have a plan for that, um, for that. Yeah, it can be really helpful for preventing cervical cancer, as, as I'm sure you'll talk to the, with the family doctor about. Yeah, and have you had any hospitalizations or any surgeries?

P: Uh no.

D: Okay, um, where about are you living right now, and who are living with?

P: I live uh, with my parents um, and my two younger siblings. We live in a house.

D: Has anybody else been sick or have similar symptoms?

P: Uh no, no one at home's been sick.

D: Okay, and uh, how about anybody at school or, or work, or anything like that that you've been around who's been sick?

P: So one of my friends actually had like a runny nose and a cough as well a couple of days ago, but uh, she seems to be better now and her voice is okay.

D: Okay, and um, have you traveled anywhere outside of the city or province?

P: No, no, not recently.

D: Alright, and, um, in the home, is there any uh like, are you exposed to any violence of any sort, like physical or emotional, either yourself or, or witnessing?

P: No.

D: Okay, and what grade are you in?

P: I'm in grade 7 or, yeah grade 7 now.

D: Oh awesome, and uh, I don, is there any smoking in the home?

P: Uh no, no one smokes at home.

D: Okay um, and then, um, anybody in the family uh, have any heart or lung conditions?

P: Heart or lung? I know, um, like, heart disease, runs on my father's side of the family. I'm not really sure like exactly what, what kind of diseases though.

D: Okay, um, and I, sometimes people that, kind of in this age group, might experience with either drugs or alcohol. Have any of your friends done that or, or yourself?

P: I know like some of my friends have uh, tried alcohol and try marijuana, but I haven't experimented yet.

D: Okay, that's good and um, I, that's all I wanted to ask today on history. Was there anything else that you wanted to add?

P: Oh, nothing that I want to add. I just, what do you, how do you think I can make this, make my voice better? And how long will I have, have this horse voice?

D: Yeah, so uh, right now it's sounding like um, a viral type illness where there's inflammation of the vocal cords, uh, with having the cough and the runny nose a couple of days ago. Actually, this reminds me that. Have you, I'm not sure if I asked about fevers or chills, if you have any?

P: Uh no, I haven't.

D: Okay, um, and so for, if it is a viral infection then, um, it's just supportive kind of management. So that will mean trying to stay as hydrated as you possibly can over these next um, next few days and then also you can use things like, if they help, like lozenges or something along those lines. Um, but it will, it will just take some time for it to come back on its own, a few days to a couple of weeks sometimes. And the cough could possibly last for um, two to four weeks after um, after having a viral type illness.

P: Okay.

D: Yeah, so it could be quite a few weeks that the symptoms last for. But since you're having a cough, and these symptoms could overlap with COVID we'll want to get a COVID swab today as well um, and kind of go from there.

P: Okay. Okay, that sounds good, thank you."""

# Execute the pipeline
if __name__ == "__main__":
    print("🏥 ADVANCED CLINICAL NER PIPELINE")
    print("=" * 60)
    print("Using pre-trained medical model for accurate entity extraction")
    print("=" * 60)

    # Process the conversation
    results = process_clinical_conversation(SAMPLE_CONVERSATION, "clinical_conversation.txt")

    # Display the clean results
    print("\n" + results['clinical_summary'])

    # Show some statistics
    print("\n" + "=" * 60)
    print("EXTRACTION DETAILS")
    print("=" * 60)
    print(f"Total entities extracted: {results['extraction_result']['num_entities']}")
    print(f"Speaker distribution: {results['extraction_result']['speaker_breakdown']}")

🏥 ADVANCED CLINICAL NER PIPELINE
Using pre-trained medical model for accurate entity extraction


config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



CLINICAL ENTITY EXTRACTION SUMMARY

🧬 PATIENT SYMPTOMS:
----------------------------------------
  • A cough
  • A runny nose
  • A sore throat
  • A viral infection
  • A viral type illness
  • A wheeze
  • Any
  • Any abdominal
  • Any allergies
  • Any blood
  • Any heart or lung conditions
  • Any loss of your taste or sense of smell
  • Any medical conditions
  • Any nausea
  • Any other symptoms
  • Any phlegm
  • Any uh,
  • Any urinary problems
  • Cervical cancer
  • Chest
  • Chill
  • Chills
  • Cough
  • Diarr
  • Discharge
  • Dry cough
  • Es
  • Eye discharge
  • Eye redness
  • Fever
  • Fevers
  • Hea
  • Heart
  • Inflammation of the vocal cords
  • Joint pains
  • Mucus
  • Muscle aches
  • Night sweats
  • Other symptoms
  • Pain
  • Rash
  • Really hoarse
  • Sick
  • Similar symptoms
  • Skin changes
  • Some weight gain
  • Sputum
  • Straining your voice
  • The
  • The cough
  • The runny nose
  • The symptoms
  • These symptoms
  • This cough
  • Vomit
  • Yo

In [7]:
import os
import sys
import json
import re
import argparse
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, Counter
from datetime import datetime
from dataclasses import dataclass
from tqdm import tqdm
import pandas as pd

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Check imports
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForTokenClassification,
        pipeline
    )
except ImportError as e:
    logger.error(f"Missing dependencies: {e}")
    print("❌ Missing dependencies. Please install:")
    print("\npip install transformers torch pandas tqdm\n")
    sys.exit(1)

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class ModelConfig:
    """Model configuration parameters"""
    # Using a pre-trained medical NER model that works well out-of-the-box
    model_name: str = "samrawal/bert-base-uncased_clinical-ner"
    # Alternative models you can try:
    # - "d4data/biomedical-ner-all"
    # - "bvanaken/clinical-ner-biobert"
    # - "emilyalsentzer/Bio_ClinicalBERT"
    max_length: int = 512
    batch_size: int = 16

class Config:
    def __init__(self, data_dir: str):
        self.project_root = Path.cwd()
        self.raw_data_dir = Path(data_dir)
        self.output_dir = self.project_root / "outputs"
        self.model_dir = self.project_root / "models"

        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        self.model_dir.mkdir(exist_ok=True)
        (self.output_dir / "summaries").mkdir(exist_ok=True)
        (self.output_dir / "entities").mkdir(exist_ok=True)
        (self.output_dir / "statistics").mkdir(exist_ok=True)
        (self.output_dir / "reports").mkdir(exist_ok=True)

        self.model = ModelConfig()

# ============================================================================
# DATA VALIDATION
# ============================================================================

class DataValidator:
    """Validate input data and configurations"""

    @staticmethod
    def validate_data_directory(data_dir: Path) -> bool:
        """Validate data directory exists and contains files"""
        if not data_dir.exists():
            logger.error(f"Data directory does not exist: {data_dir}")
            return False

        txt_files = list(data_dir.glob("*.txt"))
        if not txt_files:
            logger.error(f"No .txt files found in {data_dir}")
            return False

        logger.info(f"Found {len(txt_files)} text files in {data_dir}")
        return True

# ============================================================================
# CONVERSATION PARSER FOR ENTIRE DATASET
# ============================================================================

class ConversationParser:
    def __init__(self, config: Config):
        self.config = config
        self.validator = DataValidator()

    def parse_file(self, filepath: Path) -> Dict:
        """Parse a single conversation file"""
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read().strip()

            if not content:
                logger.warning(f"Empty file: {filepath}")
                return self._create_empty_conversation(filepath)

            lines = [line.strip() for line in content.split('\n') if line.strip()]
            turns = self._parse_lines(lines)

            conversation = {
                'filename': filepath.name,
                'filepath': str(filepath),
                'turns': turns,
                'num_turns': len(turns),
                'total_chars': len(content),
                'word_count': sum(len(turn['text'].split()) for turn in turns)
            }

            return conversation

        except Exception as e:
            logger.error(f"Error parsing {filepath}: {e}")
            return self._create_empty_conversation(filepath)

    def _parse_lines(self, lines: List[str]) -> List[Dict]:
        """Parse individual lines into conversation turns"""
        turns = []
        current_speaker = None
        current_text = []

        for line in lines:
            speaker, text = self._parse_line(line)

            if speaker:
                # Save previous turn if exists
                if current_speaker and current_text:
                    turns.append({
                        'speaker': current_speaker,
                        'text': ' '.join(current_text)
                    })

                # Start new turn
                current_speaker = speaker
                current_text = [text] if text else []
            elif current_speaker and text:
                # Continuation of current turn
                current_text.append(text)

        # Add the last turn
        if current_speaker and current_text:
            turns.append({
                'speaker': current_speaker,
                'text': ' '.join(current_text)
            })

        return turns

    def _parse_line(self, line: str) -> Tuple[Optional[str], Optional[str]]:
        """Parse a single line for speaker and text"""
        line = line.strip()

        # Match D: or P: patterns
        if line.startswith('D:'):
            return 'doctor', line[2:].strip()
        elif line.startswith('P:'):
            return 'patient', line[2:].strip()
        elif line:
            # If no speaker tag but has content, continue previous speaker
            return None, line
        else:
            return None, None

    def _create_empty_conversation(self, filepath: Path) -> Dict:
        """Create empty conversation structure for invalid files"""
        return {
            'filename': filepath.name,
            'filepath': str(filepath),
            'turns': [],
            'num_turns': 0,
            'total_chars': 0,
            'word_count': 0
        }

    def parse_all_files(self, data_dir: Path) -> List[Dict]:
        """Parse all conversation files with progress tracking"""
        if not self.validator.validate_data_directory(data_dir):
            raise ValueError(f"Invalid data directory: {data_dir}")

        files = sorted(list(data_dir.glob('*.txt')))

        conversations = []
        logger.info(f"📁 Parsing {len(files)} conversation files...")

        for filepath in tqdm(files, desc="Parsing files"):
            conversation = self.parse_file(filepath)
            if conversation['num_turns'] > 0:  # Only include non-empty conversations
                conversations.append(conversation)
            else:
                logger.warning(f"Skipping empty conversation: {filepath.name}")

        # Generate parsing statistics
        self._log_parsing_statistics(conversations)

        return conversations

    def _log_parsing_statistics(self, conversations: List[Dict]):
        """Log detailed parsing statistics"""
        total_turns = sum(c['num_turns'] for c in conversations)
        total_words = sum(c['word_count'] for c in conversations)
        doctor_turns = sum(1 for c in conversations for t in c['turns'] if t['speaker'] == 'doctor')
        patient_turns = sum(1 for c in conversations for t in c['turns'] if t['speaker'] == 'patient')

        logger.info(f"📊 Parsing Statistics:")
        logger.info(f"  • Conversations: {len(conversations)}")
        logger.info(f"  • Total turns: {total_turns}")
        logger.info(f"  • Doctor turns: {doctor_turns}")
        logger.info(f"  • Patient turns: {patient_turns}")
        logger.info(f"  • Total words: {total_words}")
        logger.info(f"  • Avg words per conversation: {total_words/len(conversations):.1f}")

# ============================================================================
# ENHANCED ENTITY POST-PROCESSOR
# ============================================================================

class ClinicalEntityPostProcessor:
    """Advanced entity cleaning and categorization for clinical text"""

    def __init__(self):
        # Map model entities to our categories
        self.entity_mapping = {
            # Symptoms
            'problem': 'SYMPTOM',
            'symptom': 'SYMPTOM',
            'sign': 'SYMPTOM',
            'disease': 'SYMPTOM',
            'disorder': 'SYMPTOM',

            # Medications
            'drug': 'MEDICATION',
            'medication': 'MEDICATION',
            'treatment': 'MEDICATION',

            # Diagnoses
            'diagnosis': 'DIAGNOSIS',
            'condition': 'DIAGNOSIS',

            # Tests
            'test': 'TEST',
            'procedure': 'TEST',

            # Body parts
            'anatomy': 'BODY_PART',
            'body_part': 'BODY_PART'
        }

        # Common clinical stop words
        self.stop_words = {
            'patient', 'doctor', 'history', 'normal', 'negative', 'positive',
            'like', 'just', 'get', 'got', 'going', 'know', 'think', 'maybe',
            'probably', 'possibly', 'seems', 'appears', 'looks', 'feels',
            'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for'
        }

        # Medical term variations and corrections
        self.medical_corrections = {
            'c/o': 'complains of',
            's/o': 'suggestive of',
            'r/o': 'rule out',
            'h/o': 'history of',
            'coughing': 'cough',
            'vomiting': 'vomit',
            'nauseous': 'nausea',
            'aching': 'ache',
            'hurts': 'pain',
            'hurting': 'pain',
            'hoarse': 'hoarseness',
            'runny': 'runny nose'
        }

    def clean_and_categorize_entities(self, raw_entities: List[Dict]) -> List[Dict]:
        """Clean and categorize entities from the NER model"""
        cleaned_entities = []

        for entity in raw_entities:
            # Clean the entity text
            entity_text = self._clean_entity_text(entity['word'])

            if not entity_text or len(entity_text) < 2:
                continue

            # Skip stop words
            if entity_text.lower() in self.stop_words:
                continue

            # Map to our entity categories
            entity_type = self._map_entity_type(entity['entity_group'])

            if entity_type:
                cleaned_entity = {
                    'word': entity_text,
                    'entity_group': entity_type,
                    'score': float(entity['score']),
                    'original_entity': entity['entity_group']
                }
                cleaned_entities.append(cleaned_entity)

        # Remove duplicates and merge similar entities
        final_entities = self._deduplicate_entities(cleaned_entities)

        return final_entities

    def _clean_entity_text(self, text: str) -> str:
        """Clean entity text"""
        # Remove subword markers and extra spaces
        cleaned = text.replace('##', '').strip()

        # Apply medical corrections
        cleaned_lower = cleaned.lower()
        if cleaned_lower in self.medical_corrections:
            cleaned = self.medical_corrections[cleaned_lower]

        # Remove common prefixes/suffixes
        cleaned = re.sub(r'^(mild|moderate|severe|acute|chronic)\s+', '', cleaned)
        cleaned = re.sub(r'\s+(disease|disorder|syndrome|pain|ache)$', '', cleaned)

        # Capitalize properly
        if len(cleaned) > 1:
            cleaned = cleaned[0].upper() + cleaned[1:]

        return cleaned.strip()

    def _map_entity_type(self, original_type: str) -> str:
        """Map model entity types to our categories"""
        original_lower = original_type.lower()

        for key, value in self.entity_mapping.items():
            if key in original_lower:
                return value

        # Default mapping based on common patterns
        if any(word in original_lower for word in ['drug', 'med', 'pill', 'tablet']):
            return 'MEDICATION'
        elif any(word in original_lower for word in ['symptom', 'pain', 'fever', 'cough']):
            return 'SYMPTOM'
        elif any(word in original_lower for word in ['diagnosis', 'disease', 'infection']):
            return 'DIAGNOSIS'
        elif any(word in original_lower for word in ['test', 'exam', 'xray', 'scan']):
            return 'TEST'
        elif any(word in original_lower for word in ['body', 'anatomy', 'organ']):
            return 'BODY_PART'

        return 'SYMPTOM'  # Default fallback

    def _deduplicate_entities(self, entities: List[Dict]) -> List[Dict]:
        """Remove duplicate and very similar entities"""
        seen = set()
        unique_entities = []

        for entity in entities:
            # Create a key based on cleaned text and type
            key = (entity['word'].lower(), entity['entity_group'])

            if key not in seen:
                seen.add(key)
                unique_entities.append(entity)

        return unique_entities

# ============================================================================
# ADVANCED INFERENCE ENGINE FOR DATASET
# ============================================================================

class ClinicalNERPipeline:
    """Advanced clinical NER pipeline using pre-trained models"""

    def __init__(self, config: Config):
        self.config = config
        self.postprocessor = ClinicalEntityPostProcessor()
        self.ner_pipeline = None
        self._load_model()

    def _load_model(self):
        """Load the pre-trained clinical NER model"""
        try:
            logger.info(f"🚀 Loading clinical NER model: {self.config.model.model_name}")

            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1,
                batch_size=self.config.model.batch_size
            )

            logger.info("✅ Clinical NER model loaded successfully!")

        except Exception as e:
            logger.error(f"❌ Error loading model: {e}")
            logger.info("🔄 Trying fallback model...")

            # Fallback to a more reliable model
            self.config.model.model_name = "emilyalsentzer/Bio_ClinicalBERT"
            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1
            )

    def extract_clinical_entities(self, conversation: Dict) -> Dict:
        """Extract clinical entities from conversation"""
        all_entities = []

        for turn in conversation['turns']:
            if not turn['text'].strip():
                continue

            try:
                # Extract entities from this turn
                raw_entities = self.ner_pipeline(turn['text'])

                # Clean and categorize entities
                cleaned_entities = self.postprocessor.clean_and_categorize_entities(raw_entities)

                # Add conversation context
                for entity in cleaned_entities:
                    entity['speaker'] = turn['speaker']
                    entity['turn_text'] = turn['text'][:100] + '...' if len(turn['text']) > 100 else turn['text']
                    entity['confidence'] = entity['score']

                all_entities.extend(cleaned_entities)

            except Exception as e:
                logger.warning(f"⚠️ Error processing turn in {conversation['filename']}: {str(e)[:100]}...")
                continue

        return {
            'filename': conversation['filename'],
            'entities': all_entities,
            'num_entities': len(all_entities),
            'speaker_breakdown': self._get_speaker_breakdown(all_entities)
        }

    def extract_entities_batch(self, conversations: List[Dict]) -> List[Dict]:
        """Extract entities from multiple conversations efficiently"""
        all_results = []

        logger.info(f"🔍 Extracting clinical entities from {len(conversations)} conversations...")

        for conversation in tqdm(conversations, desc="Processing conversations"):
            result = self.extract_clinical_entities(conversation)
            all_results.append(result)

        return all_results

    def _get_speaker_breakdown(self, entities: List[Dict]) -> Dict[str, int]:
        """Get entity count by speaker"""
        breakdown = defaultdict(int)
        for entity in entities:
            breakdown[entity.get('speaker', 'unknown')] += 1
        return dict(breakdown)

# ============================================================================
# INTELLIGENT RESULTS GENERATOR FOR DATASET
# ============================================================================

class ClinicalResultsGenerator:
    """Generate clean, meaningful clinical summaries for entire dataset"""

    @staticmethod
    def generate_detailed_results(extraction_result: Dict) -> Dict:
        """Generate detailed, organized results for a single conversation"""
        entities = extraction_result['entities']

        # Group entities by type
        symptoms = []
        medications = []
        diagnoses = []
        tests = []
        body_parts = []

        for entity in entities:
            entity_type = entity['entity_group']
            entity_text = entity['word']

            # Skip low confidence entities
            if entity.get('confidence', 0) < 0.6:
                continue

            if entity_type == 'SYMPTOM':
                symptoms.append(entity_text)
            elif entity_type == 'MEDICATION':
                medications.append(entity_text)
            elif entity_type == 'DIAGNOSIS':
                diagnoses.append(entity_text)
            elif entity_type == 'TEST':
                tests.append(entity_text)
            elif entity_type == 'BODY_PART':
                body_parts.append(entity_text)

        # Remove duplicates and sort
        symptoms = sorted(list(set(symptoms)))
        medications = sorted(list(set(medications)))
        diagnoses = sorted(list(set(diagnoses)))
        tests = sorted(list(set(tests)))
        body_parts = sorted(list(set(body_parts)))

        return {
            'symptoms': symptoms,
            'medications': medications,
            'diagnoses': diagnoses,
            'tests': tests,
            'body_parts': body_parts,
            'summary': {
                'total_entities': len(symptoms) + len(medications) + len(diagnoses) + len(tests) + len(body_parts),
                'symptoms_count': len(symptoms),
                'medications_count': len(medications),
                'diagnoses_count': len(diagnoses),
                'tests_count': len(tests),
                'body_parts_count': len(body_parts)
            }
        }

    @staticmethod
    def generate_clinical_summary(detailed_results: Dict, filename: str) -> str:
        """Generate a clean, professional clinical summary"""
        summary = []
        summary.append("=" * 70)
        summary.append("CLINICAL ENTITY EXTRACTION SUMMARY")
        summary.append("=" * 70)
        summary.append(f"File: {filename}")
        summary.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        summary.append("")

        # Patient Symptoms
        summary.append("🧬 PATIENT SYMPTOMS:")
        summary.append("-" * 40)
        symptoms = detailed_results['symptoms']
        if symptoms:
            for symptom in symptoms:
                summary.append(f"  • {symptom}")
        else:
            summary.append("  No symptoms documented")

        # Medications
        summary.append("\n💊 MEDICATIONS & TREATMENTS:")
        summary.append("-" * 40)
        medications = detailed_results['medications']
        if medications:
            for med in medications:
                summary.append(f"  • {med}")
        else:
            summary.append("  No medications mentioned")

        # Clinical Diagnoses
        summary.append("\n🏥 CLINICAL IMPRESSIONS:")
        summary.append("-" * 40)
        diagnoses = detailed_results['diagnoses']
        if diagnoses:
            for diag in diagnoses:
                summary.append(f"  • {diag}")
        else:
            summary.append("  No diagnoses documented")

        # Diagnostic Tests
        summary.append("\n🔬 DIAGNOSTIC TESTS:")
        summary.append("-" * 40)
        tests = detailed_results['tests']
        if tests:
            for test in tests:
                summary.append(f"  • {test}")
        else:
            summary.append("  No tests ordered")

        # Body Parts
        summary.append("\n📍 BODY PARTS EXAMINED:")
        summary.append("-" * 40)
        body_parts = detailed_results['body_parts']
        if body_parts:
            for part in body_parts:
                summary.append(f"  • {part}")
        else:
            summary.append("  No specific body parts mentioned")

        # Statistics
        summary.append("\n📊 EXTRACTION STATISTICS:")
        summary.append("-" * 40)
        stats = detailed_results['summary']
        summary.append(f"  Total clinical entities: {stats['total_entities']}")
        summary.append(f"  Symptoms identified: {stats['symptoms_count']}")
        summary.append(f"  Medications mentioned: {stats['medications_count']}")
        summary.append(f"  Clinical impressions: {stats['diagnoses_count']}")
        summary.append(f"  Diagnostic tests: {stats['tests_count']}")
        summary.append(f"  Body parts examined: {stats['body_parts_count']}")

        summary.append("=" * 70)
        return "\n".join(summary)

    @staticmethod
    def generate_dataset_statistics(all_results: List[Dict]) -> str:
        """Generate comprehensive statistics for the entire dataset"""
        stats_lines = []
        stats_lines.append("=" * 80)
        stats_lines.append("DATASET-WIDE CLINICAL ENTITY STATISTICS")
        stats_lines.append("=" * 80)
        stats_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        stats_lines.append(f"Total Conversations Analyzed: {len(all_results)}")
        stats_lines.append("")

        # Aggregate statistics
        total_entities = 0
        entity_type_counts = Counter()
        speaker_entity_counts = defaultdict(Counter)
        all_symptoms = []
        all_medications = []
        all_diagnoses = []

        for result in all_results:
            total_entities += result['num_entities']

            # Count by entity type
            for entity in result['entities']:
                entity_type_counts[entity['entity_group']] += 1
                speaker_entity_counts[entity['speaker']][entity['entity_group']] += 1

                # Collect specific entity types for frequency analysis
                if entity['entity_group'] == 'SYMPTOM':
                    all_symptoms.append(entity['word'].lower())
                elif entity['entity_group'] == 'MEDICATION':
                    all_medications.append(entity['word'].lower())
                elif entity['entity_group'] == 'DIAGNOSIS':
                    all_diagnoses.append(entity['word'].lower())

        # Overall statistics
        stats_lines.append("OVERALL STATISTICS:")
        stats_lines.append("-" * 40)
        stats_lines.append(f"Total clinical entities extracted: {total_entities}")
        stats_lines.append(f"Average entities per conversation: {total_entities/len(all_results):.1f}")
        stats_lines.append("")

        # Entity type distribution
        stats_lines.append("ENTITY TYPE DISTRIBUTION:")
        stats_lines.append("-" * 40)
        for entity_type, count in entity_type_counts.most_common():
            percentage = (count / total_entities) * 100
            stats_lines.append(f"  {entity_type:<15} {count:>5} ({percentage:.1f}%)")
        stats_lines.append("")

        # Speaker distribution
        stats_lines.append("ENTITY DISTRIBUTION BY SPEAKER:")
        stats_lines.append("-" * 40)
        for speaker in sorted(speaker_entity_counts.keys()):
            total_speaker_entities = sum(speaker_entity_counts[speaker].values())
            stats_lines.append(f"\n  {speaker.capitalize()} ({total_speaker_entities} entities):")
            for entity_type, count in speaker_entity_counts[speaker].most_common():
                stats_lines.append(f"    {entity_type:<15} {count:>5}")
        stats_lines.append("")

        # Most common symptoms
        stats_lines.append("TOP 10 MOST FREQUENT SYMPTOMS:")
        stats_lines.append("-" * 40)
        symptom_counts = Counter(all_symptoms)
        for symptom, count in symptom_counts.most_common(10):
            stats_lines.append(f"  {symptom:<25} {count:>5}")
        stats_lines.append("")

        # Most common medications
        stats_lines.append("TOP 10 MOST FREQUENT MEDICATIONS:")
        stats_lines.append("-" * 40)
        medication_counts = Counter(all_medications)
        for medication, count in medication_counts.most_common(10):
            stats_lines.append(f"  {medication:<25} {count:>5}")
        stats_lines.append("")

        # Most common diagnoses
        stats_lines.append("TOP 10 MOST FREQUENT DIAGNOSES:")
        stats_lines.append("-" * 40)
        diagnosis_counts = Counter(all_diagnoses)
        for diagnosis, count in diagnosis_counts.most_common(10):
            stats_lines.append(f"  {diagnosis:<25} {count:>5}")

        stats_lines.append("")
        stats_lines.append("=" * 80)
        return "\n".join(stats_lines)

# ============================================================================
# MAIN PIPELINE FOR ENTIRE DATASET
# ============================================================================

class MedicalSLUPipeline:
    """Main pipeline orchestrator for entire dataset"""

    def __init__(self, data_dir: str):
        self.config = Config(data_dir)
        self.results = []

    def run(self):
        """Execute the complete pipeline on entire dataset"""
        logger.info("🚀 Starting Medical Conversation SLU Pipeline")
        start_time = datetime.now()

        try:
            # Step 1: Parse all conversations
            conversations = self.step_parse_conversations()

            # Step 2: Extract clinical entities using pre-trained model
            extraction_results = self.step_extract_entities(conversations)

            # Step 3: Generate individual summaries and statistics
            self.step_generate_outputs(extraction_results)

            # Step 4: Generate dataset-wide statistics
            self.step_generate_dataset_report(extraction_results)

            # Final summary
            self.generate_final_summary(start_time, extraction_results)

        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            raise

    def step_parse_conversations(self) -> List[Dict]:
        """Step 1: Parse all conversations"""
        logger.info("STEP 1: PARSING CONVERSATIONS")

        parser = ConversationParser(self.config)
        conversations = parser.parse_all_files(self.config.raw_data_dir)

        if not conversations:
            raise ValueError("No valid conversations found after parsing")

        logger.info(f"✅ Successfully parsed {len(conversations)} conversations")
        return conversations

    def step_extract_entities(self, conversations: List[Dict]) -> List[Dict]:
        """Step 2: Extract clinical entities using pre-trained model"""
        logger.info("STEP 2: EXTRACTING CLINICAL ENTITIES")

        ner_pipeline = ClinicalNERPipeline(self.config)
        extraction_results = ner_pipeline.extract_entities_batch(conversations)

        # Save extraction results
        entities_file = self.config.output_dir / "entities" / "all_conversations_entities.json"
        with open(entities_file, 'w', encoding='utf-8') as f:
            json.dump(extraction_results, f, indent=2, ensure_ascii=False)

        total_entities = sum(r['num_entities'] for r in extraction_results)
        logger.info(f"✅ Extracted {total_entities} clinical entities from {len(extraction_results)} conversations")

        return extraction_results

    def step_generate_outputs(self, extraction_results: List[Dict]):
        """Step 3: Generate individual summaries and statistics"""
        logger.info("STEP 3: GENERATING CLINICAL SUMMARIES")

        # Generate individual summaries for each conversation
        logger.info("Generating individual clinical summaries...")
        for result in tqdm(extraction_results, desc="Creating summaries"):
            detailed_results = ClinicalResultsGenerator.generate_detailed_results(result)
            clinical_summary = ClinicalResultsGenerator.generate_clinical_summary(
                detailed_results, result['filename']
            )

            filename = Path(result['filename']).stem
            output_file = self.config.output_dir / "summaries" / f"{filename}_clinical_summary.txt"

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(clinical_summary)

        logger.info("✅ All clinical summaries generated successfully")

    def step_generate_dataset_report(self, extraction_results: List[Dict]):
        """Step 4: Generate dataset-wide statistics and report"""
        logger.info("STEP 4: GENERATING DATASET REPORT")

        # Generate comprehensive dataset statistics
        dataset_stats = ClinicalResultsGenerator.generate_dataset_statistics(extraction_results)

        stats_file = self.config.output_dir / "statistics" / "dataset_statistics.txt"
        with open(stats_file, 'w', encoding='utf-8') as f:
            f.write(dataset_stats)

        # Generate CSV report for easy analysis
        self._generate_csv_report(extraction_results)

        logger.info("✅ Dataset report generated successfully")

    def _generate_csv_report(self, extraction_results: List[Dict]):
        """Generate a CSV report for easy data analysis"""
        report_data = []

        for result in extraction_results:
            # Count entities by type for this conversation
            entity_counts = Counter()
            for entity in result['entities']:
                entity_counts[entity['entity_group']] += 1

            report_data.append({
                'filename': result['filename'],
                'total_entities': result['num_entities'],
                'symptoms_count': entity_counts['SYMPTOM'],
                'medications_count': entity_counts['MEDICATION'],
                'diagnoses_count': entity_counts['DIAGNOSIS'],
                'tests_count': entity_counts['TEST'],
                'body_parts_count': entity_counts['BODY_PART'],
                'doctor_entities': result['speaker_breakdown'].get('doctor', 0),
                'patient_entities': result['speaker_breakdown'].get('patient', 0)
            })

        # Create DataFrame and save as CSV
        df = pd.DataFrame(report_data)
        csv_file = self.config.output_dir / "reports" / "conversation_entities_report.csv"
        df.to_csv(csv_file, index=False)

        logger.info(f"📊 CSV report saved to: {csv_file}")

    def generate_final_summary(self, start_time: datetime, extraction_results: List[Dict]):
        """Generate final execution summary"""
        end_time = datetime.now()
        duration = end_time - start_time

        total_entities = sum(r['num_entities'] for r in extraction_results)
        total_conversations = len(extraction_results)

        summary = f"""
╔{'═' * 68}╗
║              DATASET PROCESSING COMPLETE                ║
╚{'═' * 68}╝

📊 RESULTS:
  • Conversations processed: {total_conversations}
  • Total clinical entities extracted: {total_entities}
  • Average entities per conversation: {total_entities/total_conversations:.1f}
  • Execution time: {duration}

📁 OUTPUTS:
  • Clinical Summaries: {self.config.output_dir / 'summaries'}
  • Entity Extractions: {self.config.output_dir / 'entities' / 'all_conversations_entities.json'}
  • Statistics: {self.config.output_dir / 'statistics'}
  • Reports: {self.config.output_dir / 'reports'}

✅ PIPELINE COMPLETED SUCCESSFULLY!
"""
        logger.info(summary)

        # Save summary to file
        summary_file = self.config.output_dir / "execution_summary.txt"
        with open(summary_file, 'w') as f:
            f.write(summary)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main entry point with argument parsing"""
    parser = argparse.ArgumentParser(description='Medical Conversation SLU Pipeline for Entire Dataset')
    parser.add_argument('/content/clinical_data', type=str,
                       help='Path to directory containing clinical conversation .txt files')

    args = parser.parse_args([DOWNLOAD_PATH])

    # Validate data directory
    if not Path(args.data_dir).exists():
        print(f"❌ Error: Data directory '{args.data_dir}' does not exist")
        sys.exit(1)

    try:
        # Initialize and run pipeline
        pipeline = MedicalSLUPipeline(args.data_dir)
        pipeline.run()

    except Exception as e:
        logger.error(f"Pipeline execution failed: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()


Parsing files:  20%|██        | 55/272 [00:00<00:00, 543.12it/s]WARNING:__main__:Skipping empty conversation: RES0002.txt

Parsing files: 100%|██████████| 272/272 [00:00<00:00, 965.80it/s]
Device set to use cuda:0

Creating summaries: 100%|██████████| 270/270 [00:00<00:00, 3741.71it/s]
